In [158]:
# Data Cleaning and Analsys
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats


# Data Preprocessing
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


#Evaluate model
from sklearn.metrics import confusion_matrix,classification_report,auc

from pathlib import Path
from config import get_engine

In [159]:
OUTPUT_DIR = Path('output')
DATA_DIR = Path("data")
engine = get_engine()

In [160]:
df=pd.read_csv(OUTPUT_DIR / 'cleaned_master_dataset.csv')

In [161]:
customer_ids = df['Customer_ID']

In [162]:
df.drop(columns=['Customer_ID', 'Geo_ID', 'Industry_ID','Product_ID'], inplace=True)

In [163]:
df.head()

,Cluster_Label,Account_Tier,Company_Size,Company_Age_Years,Employees,Legal_Entity_Type,HQ_vs_Billing,Tenure_Months,Contract_Type,Contract_Start_Date,...,Base_Monthly_Price_USD,Included_Licenses,API_Limit_Monthly,SLA_Uptime_Pct,Support_Level,AI_Features_Included,avg_monthly_revenue,revenue_per_employee,licenses_unused,Total_days
0,At-Risk SMB,Platinum,Startup (<50),10,91.0,Ltd,Same as Billing,46,Pay-As-You-Go,2023-05-18,...,4000,500,500000,99.9,Premium,Yes,8.652174,1.450549,6,365
1,High-Growth Startup,Gold,Startup (<50),1,309.0,AG,Different Country,28,Annual,2022-07-29,...,500,25,10000,99.0,Community,No,356.750000,127.113269,107,365
2,High Revenue Loyal,Bronze,Mid-Market (251-1000),7,990.0,GmbH,Same as Billing,63,Multi-Year,2024-08-30,...,4000,500,500000,99.9,Premium,Yes,1623.492063,69.527273,-59,730
3,At-Risk SMB,Strategic,Startup (<50),2,121.0,LLC,Different Country,44,Monthly,2023-04-21,...,500,25,10000,99.0,Community,No,28.772727,75.181818,-11,90
4,High Revenue Loyal,Bronze,SMB (50-250),9,245.0,Sdn Bhd,Same as Billing,50,Annual,2022-03-02,...,1500,100,100000,99.5,Standard,No,168.640000,57.318367,33,365


In [164]:
df.dtypes

Cluster_Label               str
Account_Tier                str
Company_Size                str
Company_Age_Years         int64
Employees               float64
                         ...   
AI_Features_Included        str
avg_monthly_revenue     float64
revenue_per_employee    float64
licenses_unused           int64
Total_days                int64
Length: 103, dtype: object

In [165]:
df['Contract_Start_Date'] = pd.to_datetime(df['Contract_Start_Date'], errors='coerce')
df['Contract_End_Date'] = pd.to_datetime(df['Contract_End_Date'], errors='coerce')
df['contract_duration'] = (
    df['Contract_End_Date'] - df['Contract_Start_Date']
).dt.days
reference_date = pd.to_datetime("today")

df['days_to_expiry'] = (df['Contract_End_Date'] - reference_date).dt.days
reference_date = df['Contract_Start_Date'].max()
df['contract_progress'] = (
    df['Tenure_Months'] / (df['contract_duration'] / 30)
)
df['near_renewal'] = (df['days_to_expiry'] <= 30).astype(int)
df['early_stage_customer'] = (df['Tenure_Months'] <= 3).astype(int)

In [166]:
df.drop(columns=['Contract_Start_Date', 'Contract_End_Date', 'Total_days'], inplace=True)

In [167]:
for col, dtype in df.dtypes.items():
    print(col, dtype)

Cluster_Label str
Account_Tier str
Company_Size str
Company_Age_Years int64
Employees float64
Legal_Entity_Type str
HQ_vs_Billing str
Tenure_Months int64
Contract_Type str
Renewal_Probability float64
CSM_Assigned str
Executive_Sponsor str
Health_Score int64
Onboarding_Completed str
QBR_Completed_Last_6M str
Last_Training_Days_Ago int64
Licenses_Purchased int64
Daily_Active_Users int64
Feature_Adoption_Pct float64
API_Calls_Monthly int64
Mobile_Users_Pct float64
Sessions_Per_User_Day float64
Avg_Session_Minutes float64
Integrations_Count int64
Custom_Reports_Count int64
Automations_Active int64
Support_Tickets int64
Escalated_Tickets int64
CSAT_Score float64
NPS_Score float64
Subscription_Value_Local int64
Discount_Pct float64
ACV_Local int64
ACV_USD int64
Last_Invoice_Amount_USD int64
Lifetime_Revenue_USD int64
Add_On_Revenue_USD int64
Payment_Delay_Days int64
Price_Sensitivity str
Competitor_Mentioned str
Downsell_Requested str
Renewal_Risk_Flag str
Churn int64
Next_Quarter_Revenue_US

In [168]:
convert_df = df.copy()

# ----------------------------
# 1. Convert YES/NO to binary
# ----------------------------
binary_map = {
    "Yes": 1, "No": 0,
    "Y": 1, "N": 0,
    True: 1, False: 0
}

binary_cols = [
    "Onboarding_Completed",
    "QBR_Completed_Last_6M",
    "Competitor_Mentioned",
    "Downsell_Requested",
    "Renewal_Risk_Flag",
    "SLA_Breached",
    "Escalated_To_Manager",
    "AI_Features_Included"
]

for col in binary_cols:
    if col in convert_df.columns:
        convert_df[col] = convert_df[col].map(binary_map).fillna(0).astype(int)

# ----------------------------
# 2. Convert categorical columns
# ----------------------------
cat_cols = [
    "Cluster_Label", "Account_Tier", "Company_Size",
    "Legal_Entity_Type", "HQ_vs_Billing", "Contract_Type",
    "CSM_Assigned", "Executive_Sponsor",
    "Price_Sensitivity", "Market_Tier", "SaaS_Maturity",
    "Industry", "Domain", "Business_Model",
    "Regulatory_Complexity", "Sector",
    "Relative_Churn_Risk", "SKU", "Plan_Type",
    "Module", "Support_Level",
    "Continent", "Country", "ISO2", "Currency"
]

for col in cat_cols:
    if col in convert_df.columns:
        convert_df[col] = convert_df[col].astype("category")

# ----------------------------
# 3. Ensure numeric columns are numeric
# ----------------------------
num_cols = convert_df.select_dtypes(include=["object"]).columns

for col in num_cols:
    try:
        convert_df[col] = pd.to_numeric(convert_df[col])
    except:
        pass  # leave real strings as category

# ----------------------------
# 4. Fix boolean numeric safety
# ----------------------------
convert_df["Employees"] = convert_df["Employees"].replace(0, np.nan)

# ----------------------------
# 5. Check missing values
# ----------------------------
missing = convert_df.isnull().sum().sort_values(ascending=False)
print("Top missing columns:\n", missing.head(20))

# ----------------------------
# 6. Final dtype check
# ----------------------------
print(convert_df.dtypes)

Top missing columns:
 Cluster_Label             0
Account_Tier              0
Company_Size              0
Company_Age_Years         0
Employees                 0
Legal_Entity_Type         0
HQ_vs_Billing             0
Tenure_Months             0
Contract_Type             0
Renewal_Probability       0
CSM_Assigned              0
Executive_Sponsor         0
Health_Score              0
Onboarding_Completed      0
QBR_Completed_Last_6M     0
Last_Training_Days_Ago    0
Licenses_Purchased        0
Daily_Active_Users        0
Feature_Adoption_Pct      0
API_Calls_Monthly         0
dtype: int64
Cluster_Label           category
Account_Tier            category
Company_Size            category
Company_Age_Years          int64
Employees                float64
                          ...   
contract_duration          int64
days_to_expiry             int64
contract_progress        float64
near_renewal               int64
early_stage_customer       int64
Length: 105, dtype: object


C:\Users\hello\AppData\Local\Temp\ipykernel_9408\3564734075.py:49: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  num_cols = convert_df.select_dtypes(include=["object"]).columns


In [169]:
from scipy.stats import chi2_contingency, mannwhitneyu
target = "Churn"

In [170]:
cat_cols = convert_df.select_dtypes(include=["category", "object"]).columns

chi_results = []

for col in cat_cols:
    if col == target:
        continue
    
    table = pd.crosstab(convert_df[col], convert_df[target])
    
    if table.shape[0] < 2:
        continue
    
    chi2, p, dof, exp = chi2_contingency(table)
    
    chi_results.append((col, p))

chi_df = pd.DataFrame(chi_results, columns=["feature", "p_value"])
chi_df = chi_df.sort_values("p_value")

print(chi_df.head(20))

C:\Users\hello\AppData\Local\Temp\ipykernel_9408\164752712.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = convert_df.select_dtypes(include=["category", "object"]).columns


              feature        p_value
0       Cluster_Label  1.640560e-166
8   Price_Sensitivity   2.404119e-13
13           Currency   2.973707e-02
10          Subregion   6.289510e-02
11            Country   6.545637e-02
12               ISO2   6.545637e-02
18     Business_Model   1.666284e-01
3   Legal_Entity_Type   1.862621e-01
22                SKU   2.008889e-01
24             Module   2.008889e-01
20             Sector   3.136200e-01
25      Support_Level   3.325393e-01
23          Plan_Type   3.325393e-01
2        Company_Size   3.455073e-01
9           Continent   3.642007e-01
6        CSM_Assigned   3.850258e-01
16           Industry   3.914721e-01
7   Executive_Sponsor   4.637650e-01
14        Market_Tier   5.428456e-01
1        Account_Tier   6.092701e-01


In [171]:
num_cols = convert_df.select_dtypes(include=["int64", "float64"]).columns
num_cols = [c for c in num_cols if c != target]

mw_results = []

for col in num_cols:
    group0 = convert_df[convert_df[target] == 0][col].dropna()
    group1 = convert_df[convert_df[target] == 1][col].dropna()
    
    if len(group0) < 10 or len(group1) < 10:
        continue
    
    stat, p = mannwhitneyu(group0, group1, alternative="two-sided")
    
    mw_results.append((col, p))

mw_df = pd.DataFrame(mw_results, columns=["feature", "p_value"])
mw_df = mw_df.sort_values("p_value")

print(mw_df.head(20))

                      feature        p_value
48      Net_Revenue_USD_count  2.011218e-233
33   Next_Quarter_Revenue_USD  1.646984e-175
37  Feature_Adoption_Pct_mean  3.473407e-119
38  Feature_Adoption_Pct_last  5.899712e-119
41     API_Calls_Monthly_mean  4.979631e-116
11          API_Calls_Monthly  1.804531e-114
19          Escalated_Tickets  2.553104e-112
24                  ACV_Local  2.692595e-112
17         Automations_Active  5.072163e-111
3         Renewal_Probability  1.889775e-110
34    Daily_Active_Users_mean  3.288019e-110
22   Subscription_Value_Local  6.019459e-110
36    Daily_Active_Users_last  9.872882e-110
4                Health_Score  9.992329e-107
10       Feature_Adoption_Pct  9.870528e-106
18            Support_Tickets  2.053557e-104
9          Daily_Active_Users  6.306743e-103
16       Custom_Reports_Count  1.550522e-101
14        Avg_Session_Minutes  2.199709e-100
21                  NPS_Score  2.584087e-100


In [172]:
sig_cat = chi_df[chi_df["p_value"] < 0.05]["feature"].tolist()
sig_num = mw_df[mw_df["p_value"] < 0.05]["feature"].tolist()

selected_features = sig_cat + sig_num

print("Total selected features:", len(selected_features))
print(selected_features)

Total selected features: 59
['Cluster_Label', 'Price_Sensitivity', 'Currency', 'Net_Revenue_USD_count', 'Next_Quarter_Revenue_USD', 'Feature_Adoption_Pct_mean', 'Feature_Adoption_Pct_last', 'API_Calls_Monthly_mean', 'API_Calls_Monthly', 'Escalated_Tickets', 'ACV_Local', 'Automations_Active', 'Renewal_Probability', 'Daily_Active_Users_mean', 'Subscription_Value_Local', 'Daily_Active_Users_last', 'Health_Score', 'Feature_Adoption_Pct', 'Support_Tickets', 'Daily_Active_Users', 'Custom_Reports_Count', 'Avg_Session_Minutes', 'NPS_Score', 'Daily_Active_Users_std', 'API_Calls_Monthly_sum', 'Licenses_Purchased', 'CSAT_Score', 'payment_reliability_score', 'Payment_Delay_Days_mean', 'Payment_Delay_Days_max', 'Payment_Delay_Days', 'Sessions_Total_sum', 'Sessions_Per_User_Day', 'Integrations_Count', 'positive_sentiment_ratio', 'total_events', 'Net_Revenue_USD_sum', 'Employees', 'Total_Billed_USD_sum', 'Score', 'Add_On_Revenue_USD', 'avg_monthly_revenue', 'Last_Training_Days_Ago', 'Add_On_Revenue_U

In [173]:
Num_cont_Transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [174]:
Cat_Transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [175]:
Preprocess_Step = ColumnTransformer(transformers=[
    ('num_cont', Num_cont_Transformer, sig_num),
    ('cat', Cat_Transformer, sig_cat)
])

In [176]:
Preprocess_Step = ColumnTransformer(transformers=[
    ('num_cont', Num_cont_Transformer, sig_num),
    ('cat', Cat_Transformer, sig_cat)
])

In [177]:
# model = RandomForestClassifier(
#     n_estimators=300,
#     random_state=42,
#     class_weight="balanced"
# )
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)
Preprocess_Pipeline = Pipeline(steps=[
    ('preprocess', Preprocess_Step),
    ('model', model)
])

In [178]:
customer_ids_train, customer_ids_test = train_test_split(
    customer_ids,
    test_size=0.2,
    random_state=42,
    stratify=convert_df[target]
)

In [179]:
X_train, X_test, y_train, y_test = train_test_split(
    convert_df[sig_num + sig_cat], convert_df[target],
    test_size=0.2,
    random_state=42,
    stratify=convert_df[target]
)

In [180]:
Preprocess_Pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](59,)","['Net_Revenue_USD_count','Next_Quarter_Revenue_USD', 'Feature_Adoption_Pct_mean',...,'Cluster_Label','Price_Sensitivity', 'Currency']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,59
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_cont', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (defau

In [181]:
y_train_pred = Preprocess_Pipeline.predict(X_train)
y_test_pred = Preprocess_Pipeline.predict(X_test)

In [182]:
print("TRAIN CONFUSION MATRIX")
print(confusion_matrix(y_train, y_train_pred))

print("\nTEST CONFUSION MATRIX")
print(confusion_matrix(y_test, y_test_pred))

print("\nTRAIN REPORT")
print(classification_report(y_train, y_train_pred))

print("\nTEST REPORT")
print(classification_report(y_test, y_test_pred))

TRAIN CONFUSION MATRIX
[[3627    0]
 [   0  464]]

TEST CONFUSION MATRIX
[[907   0]
 [  4 112]]

TRAIN REPORT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3627
           1       1.00      1.00      1.00       464

    accuracy                           1.00      4091
   macro avg       1.00      1.00      1.00      4091
weighted avg       1.00      1.00      1.00      4091


TEST REPORT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       907
           1       1.00      0.97      0.98       116

    accuracy                           1.00      1023
   macro avg       1.00      0.98      0.99      1023
weighted avg       1.00      1.00      1.00      1023



In [183]:
from sklearn.metrics import roc_auc_score

y_prob = Preprocess_Pipeline.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

ROC-AUC: 0.9999049538075505


In [184]:
import pickle

In [185]:
with open('churn_prediction_model.pkl', 'wb') as f:
    pickle.dump(Preprocess_Pipeline, f)

print("Model saved successfully!")

Model saved successfully!


In [186]:
# Train probabilities
train_prob = Preprocess_Pipeline.predict_proba(X_train)[:, 1]

train_df = pd.DataFrame({
    'Customer_ID': customer_ids_train.values,
    'Churn_Probability': train_prob,
    'Dataset': 'Train'
})

# Test probabilities
test_prob = Preprocess_Pipeline.predict_proba(X_test)[:, 1]

test_df = pd.DataFrame({
    'Customer_ID': customer_ids_test.values,
    'Churn_Probability': test_prob,
    'Dataset': 'Test'
})

# Combine both DataFrames
output_df = pd.concat([train_df, test_df], ignore_index=True)

# Save to CSV
output_df.to_csv('customer_churn_probability.csv', index=False)

print(output_df.head())
print(f"Total Records: {len(output_df)}")

   Customer_ID  Churn_Probability Dataset
0        14738           0.000043   Train
1        14554           0.000237   Train
2        13237           0.000460   Train
3        11773           0.000064   Train
4        13793           0.000066   Train
Total Records: 5114
